In [26]:
import pandas as pd
import os
import re
import ast
import string
#!pip install nltk
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from tqdm import tqdm
#!pip install contractions
from contractions import contractions_dict  
#!pip install emoji
import emoji
#!pip install vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from collections import defaultdict
import numpy as np

# Emolex total words are inaccurate bc some words have 0, proportions above 1 are also inaccurate
# pos neg and neu are also inaccurate in final_df here, but i fixed it in sentiment (check before rerunning, and check after)
# some words can have multiple emotions
# add keyword flag

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\rsanc\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\rsanc\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rsanc\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\rsanc\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\rsanc\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## Load and clean the data - as of 7/22

In [27]:
# Function to clean the content based on the keywords
def clean_content(content):
    # Remove content after specific keywords including the keywords
    keywords = ["Comments are closed", "Choked by their cap", "Print"]
    for keyword in keywords:
        last_occurrence = content.rfind(keyword)
        if last_occurrence != -1:
            content = content[:last_occurrence]

    # Remove " / Daily Nexus" and the two words before it
    pattern_daily_nexus = r'\b\w+\b \b\w+\b / Daily Nexus'
    content = re.sub(pattern_daily_nexus, '', content)

    # Remove the phrase "Nexus File Photo"
    content = content.replace("Nexus File Photo", "")

    # Clean special character 
    content = content.replace('', "")

    # Remove "Escrito por:" and the two words following it
    pattern_escrito_por = r'Escrito por: \b\w+\b \b\w+\b'
    content = re.sub(pattern_escrito_por, '', content)

    return content.strip()

# Function to read CSV with different encodings
def read_csv_with_encoding(file_path):
    encodings = ['utf-8', 'latin1', 'iso-8859-1']
    for encoding in encodings:
        try:
            return pd.read_csv(file_path, encoding=encoding)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError(f"Unable to decode the file: {file_path}")

# Combine sections and keep data from the first row among duplicates
def combine_rows(group):
    combined_section = list(group['Section'].unique())
    first_row = group.iloc[0].copy()
    first_row['Section'] = combined_section
    return first_row

# Function to clean the 'Authors' column
def clean_authors(authors_str):
    try:
        # Convert the string representation of the list into an actual list
        authors_list = ast.literal_eval(authors_str)
        # Ensure it's a list and remove single quotes if present
        if isinstance(authors_list, list):
            cleaned_list = [author.replace("'", "").strip() for author in authors_list]
            return cleaned_list
    except (ValueError, SyntaxError):
        pass
    return authors_str  # Return the original string if conversion fails

In [28]:
# Path to the directory containing the CSV files
dir_path = 'scraped_files'

# List all files in the directory
files = [f for f in os.listdir(dir_path) if f.endswith('.csv')]

# Read and concatenate all CSV files
df_list = []
for file in files:
    file_path = os.path.join(dir_path, file)
    df = read_csv_with_encoding(file_path)
    df_list.append(df)

# Concatenate all dataframes
combined_df = pd.concat(df_list, ignore_index=True)

# Keep Only 0 - 6 columns
combined_df = combined_df.iloc[:, 0:7]

# Remove dirty values
combined_df = combined_df[combined_df['Section'].map(combined_df['Section'].value_counts()) > 1]

# Convert Date and Time to datetime
combined_df['Date'] = pd.to_datetime(combined_df['Date'])
combined_df['Time'] = pd.to_datetime(combined_df['Time']).dt.time

# Define the mapping from old section names to formal names
section_mapping = {
    'news': 'News',
    'sports': 'Sports',
    'artsweek': 'Artsweek',
    'daily-stench': 'Daily Stench',
    'opinion': 'Opinion',
    'on-the-menu': 'On The Menu',
    'science': 'Science & Tech',
    'photo': 'Photo',
    'video': 'Video',
    'data': 'Data',
    'games': 'Games',
    'la-vista': 'La Vista'
}

# Replace the section names with the formal names
combined_df['Section'] = combined_df['Section'].replace(section_mapping)


# In content, remove just the words after the last "Print" in the content, "Comments are closed", or "Choked by their cap"
combined_df['Content'] = combined_df['Content'].apply(clean_content)

# Apply the cleaning function to the 'Authors' column
combined_df['Authors'] = combined_df['Authors'].apply(clean_authors)

# Group by URL and apply the combination function
combined_df = combined_df.groupby('URL', as_index=False).apply(combine_rows).reset_index(drop=True)

# Drop original duplicates from the DataFrame
combined_df = combined_df.drop_duplicates('URL', keep='first')

# Save the combined dataframe to a new CSV file if needed
combined_df.to_csv('data_storage/combined_df.csv', index=False)

# Show all columns in pandas
pd.set_option('display.max_columns', None)

combined_df

C:\Users\rsanc\AppData\Local\Temp\ipykernel_25956\884806580.py:25: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  combined_df['Time'] = pd.to_datetime(combined_df['Time']).dt.time
C:\Users\rsanc\AppData\Local\Temp\ipykernel_25956\884806580.py:54: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  combined_df = combined_df.groupby('URL', as_index=False).apply(combine_rows).reset_index(drop=True)


,Section,Title,Date,Time,Authors,URL,Content
0,[Daily Stench],Decade Recap: Your 2010 Self Grossed Out You’v...,2020-01-01,00:00:00,[Sam Penn],https://dailynexus.com/2020-01-01/decade-recap...,"Using advanced, highly sensitive technology th..."
1,[News],Thirty Meter Telescope Project Now To Be Pause...,2020-01-01,16:04:00,[Max Abrams],https://dailynexus.com/2020-01-01/thirty-meter...,Construction for the Thirty Meter Telescope at...
2,[Daily Stench],"Fuck You, Flavored Cream Cheese",2020-01-02,04:20:00,[Jenny De La Cuadra],https://dailynexus.com/2020-01-02/fuck-you-fla...,Straight up: flavored cream cheese can fuck al...
3,[On The Menu],A Local’s Guide To the NYC Food Scene,2020-01-03,14:27:00,[Sonya Raytburg],https://dailynexus.com/2020-01-03/a-locals-gui...,Travelling to New York City is already an over...
4,[News],“Pattern of Burglaries” Occur in Manzanita Res...,2020-01-06,19:46:00,[Evelyn Spence],https://dailynexus.com/2020-01-06/pattern-of-b...,A “pattern of burglaries” occurred in Manzanit...
...,...,...,...,...,...,...,...
4913,[Daily Stench],5 easy side-hustles at UCSB (get rich quick!!!),2025-09-07,17:31:00,[Gentle Giant],https://dailynexus.com/2025-09-07/5-easy-side-...,Nick Throgmorton/Daily Nexus With college tuit...
4914,[La Vista],El sueño americano: un mito,2025-09-07,10:59:00,[Azucena Sanchez],https://dailynexus.com/2025-09-07/el-sueno-ame...,CORTESIA DE DREAMSTIME Hace poco vi una pelícu...
4915,[La Vista],Estudiante de UCSB arrestado por agresión sexual,2025-09-07,10:47:00,"[Michelle Cisneros, Nestor Manzanares]",https://dailynexus.com/2025-09-07/estudiante-d...,Advertencia: Este artículo contiene contenido ...
4916,[News],Discover the Best R2PBET Casino Games: A Compl...,2025-09-11,00:54:00,[Daily Nexus Photography],https://dailynexus.com/2025-09-11/discover-the...,"When exploring online casinos, the R2PBET Casi..."


## Processing

In [ ]:
# deletes spam casino posts from hacking
combined_df = combined_df.drop([4916, 4917]) 

# Function to expand contractions
def expand_contractions(text, contractions_dict=contractions_dict):
    contractions_pattern = re.compile('({})'.format('|'.join(contractions_dict.keys())), 
                                      flags=re.IGNORECASE|re.DOTALL)
    def expand_match(contraction):
        match = contraction.group(0)
        expanded_contraction = contractions_dict.get(match.lower())
        return expanded_contraction

    expanded_text = contractions_pattern.sub(expand_match, text)
    return expanded_text

# Function to remove URLs, mentions, hashtags, and special characters
def clean_text(text):
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)  # Remove URLs
    text = re.sub(r'\@\w+|\#','', text)  # Remove mentions and hashtags
    text = re.sub(r'[^A-Za-z0-9\sáéíóúñÁÉÍÓÚÑ]', '', text)  # Remove special characters except accented characters
    text = emoji.demojize(text)  # Convert emojis to text
    return text

# Function to preprocess the text in English
def preprocess_text(text):
    if isinstance(text, str):
        # Replace hyphens with spaces
        text = text.replace('-', ' ')
        # Clean text
        text = clean_text(text)
        # Expand contractions
        text = expand_contractions(text)
        # Convert to lowercase
        text = text.lower()
        # Remove punctuation
        text = text.translate(str.maketrans('', '', string.punctuation))
        # Tokenize the text
        tokens = word_tokenize(text)
        # Remove stopwords and numbers
        tokens = [word for word in tokens if word.isalpha() and word not in stopwords.words('english')]
        # Lemmatization
        lemmatizer = WordNetLemmatizer()
        tokens = [lemmatizer.lemmatize(word) for word in tokens]
        # Remove extra whitespace
        text = ' '.join(tokens).strip()
        return text
    else:
        return ''

# Function to preprocess the text in Spanish
def preprocess_text_spanish(text):
    if isinstance(text, str):
        # Replace hyphens with spaces
        text = text.replace('-', ' ')
        # Clean text
        text = clean_text(text)
        # Convert to lowercase
        text = text.lower()
        # Remove punctuation
        text = text.translate(str.maketrans('', '', string.punctuation))
        # Tokenize the text
        tokens = word_tokenize(text)
        # Remove stopwords and numbers
        spanish_stopwords = stopwords.words('spanish')
        tokens = [word for word in tokens if word.isalpha() and word not in spanish_stopwords]
        # Lemmatization is optional and complex for Spanish without proper libraries
        text = ' '.join(tokens).strip()
        return text
    else:
        return ''

In [30]:
# Load data
nlp_df = pd.read_csv('data_storage/combined_df.csv')
nlp_df['Authors'] = nlp_df['Authors'].apply(clean_authors)
nlp_df['Section'] = nlp_df['Section'].apply(clean_authors)

# Check if title and content is Spanish
nlp_df['is_spanish'] = nlp_df['Section'].apply(pd.Series).stack().str.contains('La Vista', case=False).groupby(level=0).any()

# Applying the preprocess_text function to the 'Content' column
tqdm.pandas()

# Preprocess Title with progress bar
nlp_df['Processed_Title'] = nlp_df.progress_apply(lambda x: preprocess_text_spanish(x['Title']) if x['is_spanish'] else preprocess_text(x['Title']), axis=1)

# Preprocess Content with progress bar
nlp_df['Processed_Content'] = nlp_df.progress_apply(lambda x: preprocess_text_spanish(x['Content']) if x['is_spanish'] else preprocess_text(x['Content']), axis=1)

# Save the preprocessed DataFrame to a new CSV file if needed
nlp_df.to_csv('data_storage/nlp_df.csv', index=False)

nlp_df

100%|██████████| 4918/4918 [15:22<00:00,  5.33it/s]


,Section,Title,Date,Time,Authors,URL,Content,is_spanish,Processed_Title,Processed_Content
0,[Daily Stench],Decade Recap: Your 2010 Self Grossed Out You’v...,2020-01-01,00:00:00,[Sam Penn],https://dailynexus.com/2020-01-01/decade-recap...,"Using advanced, highly sensitive technology th...",False,de recap self grossed youve done butt stuff,using advanced highly sensitive technology won...
1,[News],Thirty Meter Telescope Project Now To Be Pause...,2020-01-01,16:04:00,[Max Abrams],https://dailynexus.com/2020-01-01/thirty-meter...,Construction for the Thirty Meter Telescope at...,False,thirty meter telescope project paused uary,construction thirty meter telescope mauna kea ...
2,[Daily Stench],"Fuck You, Flavored Cream Cheese",2020-01-02,04:20:00,[Jenny De La Cuadra],https://dailynexus.com/2020-01-02/fuck-you-fla...,Straight up: flavored cream cheese can fuck al...,False,fuck flavored cream cheese,straight flavored cream cheese fuck way simply...
3,[On The Menu],A Local’s Guide To the NYC Food Scene,2020-01-03,14:27:00,[Sonya Raytburg],https://dailynexus.com/2020-01-03/a-locals-gui...,Travelling to New York City is already an over...,False,local guide nyc food scene,travelling new york city already overwhelming ...
4,[News],“Pattern of Burglaries” Occur in Manzanita Res...,2020-01-06,19:46:00,[Evelyn Spence],https://dailynexus.com/2020-01-06/pattern-of-b...,A “pattern of burglaries” occurred in Manzanit...,False,pattern burglary occur manzanita residence hal...,pattern burglary occurred manzanita village re...
...,...,...,...,...,...,...,...,...,...,...
4913,[Daily Stench],5 easy side-hustles at UCSB (get rich quick!!!),2025-09-07,17:31:00,[Gentle Giant],https://dailynexus.com/2025-09-07/5-easy-side-...,Nick Throgmorton/Daily Nexus With college tuit...,False,easy side hustle ucsb get rich quick,nick throgmortondaily nexus college tuition is...
4914,[La Vista],El sueño americano: un mito,2025-09-07,10:59:00,[Azucena Sanchez],https://dailynexus.com/2025-09-07/el-sueno-ame...,CORTESIA DE DREAMSTIME Hace poco vi una pelícu...,True,sueño americano mito,cortesia dreamstime hace vi película titulada ...
4915,[La Vista],Estudiante de UCSB arrestado por agresión sexual,2025-09-07,10:47:00,"[Michelle Cisneros, Nestor Manzanares]",https://dailynexus.com/2025-09-07/estudiante-d...,Advertencia: Este artículo contiene contenido ...,True,estudiante ucsb arrestado agresión sexual,advertencia artículo contiene contenido relaci...
4916,[News],Discover the Best R2PBET Casino Games: A Compl...,2025-09-11,00:54:00,[Daily Nexus Photography],https://dailynexus.com/2025-09-11/discover-the...,"When exploring online casinos, the R2PBET Casi...",False,discover best casino game complete review guide,exploring online casino casino stand diverse r...


In [32]:


# Load data
nlp_df = pd.read_csv('data_storage/nlp_df.csv')
nlp_df['Authors'] = nlp_df['Authors'].apply(clean_authors)
nlp_df['Section'] = nlp_df['Section'].apply(clean_authors)
translated_df = pd.read_csv('scraped_files/translated-la-vista.csv')

# delete all rows after 115
translated_df = translated_df.drop(translated_df.index[116:])
translated_df.to_csv('data_storage/temp_translated-la-vista.csv', index=False)

# Merge the two DataFrames on Title but drop Section_y and change all NaN to None and rename Section_x to section
nlp_df = nlp_df.merge(translated_df, on='Title', how='left')

# Fix merging errors -  merge the two DataFrames on Title  and content- with manual index merging
# Merge nlp_df 5590 with translated_df 47, nlp_df 5594 with translated_df 49, nlp_df 5839 with translated_df 75, nlp_df 5880 with translated_df 80, nlp_df 5929 with translated_df 88, nlp_df 6269 with translated_df 98

# Title
nlp_df.loc[5590, 'English_Title'] = translated_df.loc[47, 'English_Title']
nlp_df.loc[5594, 'English_Title'] = translated_df.loc[49, 'English_Title']
nlp_df.loc[5839, 'English_Title'] = translated_df.loc[75, 'English_Title']
nlp_df.loc[5880, 'English_Title'] = translated_df.loc[80, 'English_Title']
nlp_df.loc[5929, 'English_Title'] = translated_df.loc[88, 'English_Title']
nlp_df.loc[6269, 'English_Title'] = translated_df.loc[98, 'English_Title']

# Content
nlp_df.loc[5590, 'Translated_Content'] = translated_df.loc[47, 'Translated_Content']
nlp_df.loc[5594, 'Translated_Content'] = translated_df.loc[49, 'Translated_Content']
nlp_df.loc[5839, 'Translated_Content'] = translated_df.loc[75, 'Translated_Content']
nlp_df.loc[5880, 'Translated_Content'] = translated_df.loc[80, 'Translated_Content']
nlp_df.loc[5929, 'Translated_Content'] = translated_df.loc[88, 'Translated_Content']
nlp_df.loc[6269, 'Translated_Content'] = translated_df.loc[98, 'Translated_Content']


nlp_df.drop(columns=['Section_y'], inplace=True)
nlp_df.fillna('None', inplace=True)
nlp_df.rename(columns={'Section_x': 'Section'}, inplace=True)

# Save to a new CSV file
nlp_df.to_csv('data_storage/translated_nlp_df.csv', index=False)

nlp_df

KeyError: 'English_Title'

## VADER Sentiment Analysis

In [ ]:
# Function to analyze sentiment
def analyze_sentiment_vader(text):
    return analyzer.polarity_scores(text)

# Function to classify sentiment
def classify_sentiment(compound):
    if compound >= 0.05:
        return 'Positive'
    elif compound <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

In [ ]:
# Load data
sentiment_df = pd.read_csv('data_storage/translated_nlp_df.csv')
sentiment_df['Authors'] = sentiment_df['Authors'].apply(clean_authors)
sentiment_df['Section'] = sentiment_df['Section'].apply(clean_authors)

# Initialize the VADER sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

# Apply the sentiment analysis to the original 'Title' and 'Content'
tqdm.pandas()

# Apply sentiment analysis for Titles and Content
sentiment_df['Title_Sentiment'] = sentiment_df.progress_apply(
    lambda x: analyze_sentiment_vader(x['English_Title']) if x['is_spanish'] else analyze_sentiment_vader(x['Title']), axis=1)

sentiment_df['Content_Sentiment'] = sentiment_df.progress_apply(
    lambda x: analyze_sentiment_vader(x['Translated_Content']) if x['is_spanish'] else analyze_sentiment_vader(x['Content']), axis=1)

# Extract the sentiment scores into separate columns for Titles
sentiment_df[['Title_Neg', 'Title_Neu', 'Title_Pos', 'Title_Compound']] = sentiment_df['Title_Sentiment'].apply(pd.Series)

# Extract the sentiment scores into separate columns for Content
sentiment_df[['Content_Neg', 'Content_Neu', 'Content_Pos', 'Content_Compound']] = sentiment_df['Content_Sentiment'].apply(pd.Series)

# Drop the intermediate sentiment columns
sentiment_df.drop(columns=['Title_Sentiment', 'Content_Sentiment'], inplace=True)

sentiment_df

In [ ]:
# Use compound scores to classify overall sentiment of Title and Content
sentiment_df['Title_Sentiment'] = sentiment_df['Title_Compound'].apply(classify_sentiment)
sentiment_df['Content_Sentiment'] = sentiment_df['Content_Compound'].apply(classify_sentiment)

# Save the updated dataframe to a new CSV file
sentiment_df.to_csv('data_storage/original_sentiment.csv', index=False)

sentiment_df

In [ ]:
sentiment_df['Content_Sentiment'].value_counts()

## NRC Emotion Lexicon Analysis

In [ ]:
# Function to analyze emotions using NRC Lexicon
def analyze_emotions(text, lang='en'):
    emotion = NRCLex(text, language=lang)
    emotion_scores = emotion.affect_frequencies
    return emotion_scores

In [ ]:
# Load the data
emotion_df = pd.read_csv('data_storage/original_sentiment.csv')
emotion_df['Authors'] = emotion_df['Authors'].apply(clean_authors)
emotion_df['Section'] = emotion_df['Section'].apply(clean_authors)

emolex_df = pd.read_csv("NRC-Emotion-Lexicon/OneFilePerLanguage/Spanish-NRC-EmoLex.txt", sep='\t', keep_default_na=False)
emoint_df = pd.read_csv('NRC-Emotion-Intensity-Lexicon/OneFilePerLanguage/Spanish-NRC-Emotion-Intensity-Lexicon-v1.txt', sep='\t', keep_default_na=False)
emovad_df = pd.read_csv("NRC-VAD-Lexicon/OneFilePerLanguage/Spanish-NRC-VAD-Lexicon.txt", sep='\t', keep_default_na=False)

emolex_df

In [ ]:
all_emotions = ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'negative', 'positive', 'sadness', 'surprise', 'trust']  

# Function to get emotion scores from the EmoLex
def get_emolex_scores(word, is_spanish):
    if is_spanish:
        scores = emolex_df[emolex_df['Spanish Word'].str.lower() == word.lower()]
    else:
        scores = emolex_df[emolex_df['English Word'].str.lower() == word.lower()]
    if not scores.empty:
        return scores.iloc[0].to_dict()
    return None

# Function to get emotion intensity scores from the EmoInt
def get_emoint_scores(word, is_spanish):
    if is_spanish:
        scores = emoint_df[emoint_df['Spanish Word'].str.lower() == word.lower()]
    else:
        scores = emoint_df[emoint_df['English Word'].str.lower() == word.lower()]
    if not scores.empty:
        return scores.iloc[0].to_dict()
    return None

# Function to get VAD scores from the EmoVAD
def get_emovad_scores(word, is_spanish):
    if is_spanish:
        scores = emovad_df[emovad_df['Spanish Word'].str.lower() == word.lower()] 
    else:
        scores = emovad_df[emovad_df['English Word'].str.lower() == word.lower()]
    if not scores.empty:
        return scores.iloc[0].to_dict()
    return None

# Fisher's Z transformation
def fisher_z(r):
    return np.arctanh(r)

# Inverse Fisher's Z transformation
def inverse_fisher_z(z):
    return np.tanh(z)

# Function to aggregate scores for a list of words
def aggregate_scores(words, is_spanish):
    # Initialize all emotions with 0
    emotion_scores = {emotion: 0 for emotion in all_emotions}
    intensity_scores = {emotion: 0 for emotion in all_emotions}
    vad_scores = {'Valence': 0.0, 'Arousal': 0.0, 'Dominance': 0.0}
    vad_count = 0
    emolex_count = 0
    emoint_count = 0
    total_word_count = len(words)
    for word in words:
        emolex = get_emolex_scores(word, is_spanish)
        emoint = get_emoint_scores(word, is_spanish)
        emovad = get_emovad_scores(word, is_spanish)
        
        if emolex:
            emolex_count += 1
            for emotion, score in emolex.items():
                if emotion in all_emotions:
                    emotion_scores[emotion] += score

        if emoint:
            emoint_count += 1
            if emoint['Emotion'] in all_emotions:
                intensity_scores[emoint['Emotion']] += emoint['Emotion-Intensity-Score']

        if emovad:
            vad_scores['Valence'] += emovad['Valence']
            vad_scores['Arousal'] += emovad['Arousal']
            vad_scores['Dominance'] += emovad['Dominance']
            vad_count += 1
    # Ensure no NaN values are present in the calculations
    emotion_scores = {k: 0 if pd.isna(v) else v for k, v in emotion_scores.items()}
    intensity_scores = {k: 0 if pd.isna(v) else v for k, v in intensity_scores.items()}
    vad_scores = {k: 0.0 if pd.isna(v) else v for k, v in vad_scores.items()}

    # Calculate averages for VAD and EmoInt using Fisher's Z transformation
    if vad_count > 0:
        vad_scores_z = {k: fisher_z(vad_scores[k] / vad_count) for k in vad_scores}
        vad_scores_avg = {k: inverse_fisher_z(v) for k, v in vad_scores_z.items()}
    else:
        vad_scores_avg = {k: 0.0 for k in vad_scores}

    if emoint_count > 0:
        intensity_scores_z = {k: fisher_z(intensity_scores[k] / emoint_count) for k in intensity_scores}
        intensity_scores_avg = {k: inverse_fisher_z(v) for k, v in intensity_scores_z.items()}
    else:
        intensity_scores_avg = {k: 0.0 for k in intensity_scores}

    # Calculate averages over total words using Fisher's Z transformation
    vad_scores_total_z = {k: fisher_z(vad_scores[k] / total_word_count) for k in vad_scores}
    vad_scores_total_avg = {f"{k}_total_avg": inverse_fisher_z(v) for k, v in vad_scores_total_z.items()}

    intensity_scores_total_z = {k: fisher_z(intensity_scores[k] / total_word_count) for k in intensity_scores}
    intensity_scores_total_avg = {f"{k}_total_avg": inverse_fisher_z(v) for k, v in intensity_scores_total_z.items()}

    emotion_scores_total_avg = {f"{k}_total_avg": emotion_scores[k] / total_word_count for k in emotion_scores}
    
    return (
        emotion_scores, intensity_scores, vad_scores_avg, 
        emolex_count, total_word_count, 
        vad_scores_total_avg, intensity_scores_total_avg, emotion_scores_total_avg
    )

# Add a progress bar
tqdm.pandas()

# Apply aggregation to Processed_Title only
(
    emotion_df['Title_Emotion_Scores'],
    emotion_df['Title_Intensity_Scores'],
    emotion_df['Title_VAD_Scores'],
    emotion_df['Title_EmoLex_Words'],
    emotion_df['Title_Total_Words'],
    emotion_df['Title_VAD_Total_Avg'],
    emotion_df['Title_Intensity_Total_Avg'],
    emotion_df['Title_Emotion_Total_Avg']
) = zip(*emotion_df.progress_apply(lambda x: aggregate_scores(x['Processed_Title'].split(), x['is_spanish']), axis=1))

# Expand Title_Emotion_Scores into separate columns
emotion_df = pd.concat([emotion_df.drop(['Title_Emotion_Scores'], axis=1), emotion_df['Title_Emotion_Scores'].progress_apply(pd.Series)], axis=1)

# Expand Title_Intensity_Scores into separate columns and add '_intensity' suffix
intensity_scores_df = emotion_df['Title_Intensity_Scores'].progress_apply(pd.Series).add_suffix('_intensity')
emotion_df = pd.concat([emotion_df.drop(['Title_Intensity_Scores'], axis=1), intensity_scores_df], axis=1)

# Expand Title_VAD_Scores into separate columns
vad_scores_df = emotion_df['Title_VAD_Scores'].progress_apply(pd.Series)
emotion_df = pd.concat([emotion_df.drop(['Title_VAD_Scores'], axis=1), vad_scores_df], axis=1)

# Expand Total Averages for VAD, Intensity, and Emotion counts
vad_total_avg_df = emotion_df['Title_VAD_Total_Avg'].progress_apply(pd.Series)
emotion_df = pd.concat([emotion_df.drop(['Title_VAD_Total_Avg'], axis=1), vad_total_avg_df], axis=1)

intensity_total_avg_df = emotion_df['Title_Intensity_Total_Avg'].progress_apply(pd.Series)
emotion_df = pd.concat([emotion_df.drop(['Title_Intensity_Total_Avg'], axis=1), intensity_total_avg_df], axis=1)

emotion_total_avg_df = emotion_df['Title_Emotion_Total_Avg'].progress_apply(pd.Series)
emotion_df = pd.concat([emotion_df.drop(['Title_Emotion_Total_Avg'], axis=1), emotion_total_avg_df], axis=1)

# Save the updated dataframe to a new CSV file
emotion_df.to_csv('data_storage/full_sentiment.csv', index=False)

emotion_df

In [ ]:
# Load the data
emotion_df = pd.read_csv('data_storage/full_sentiment.csv')
emotion_df['Authors'] = emotion_df['Authors'].apply(clean_authors)
emotion_df['Section'] = emotion_df['Section'].apply(clean_authors)

# Function to aggregate scores for a list of words from content
def aggregate_content_scores(words, is_spanish):
    # Initialize all emotions with 0
    emotion_scores = {emotion: 0 for emotion in all_emotions}
    intensity_scores = {emotion: 0 for emotion in all_emotions}
    vad_scores = {'Valence': 0.0, 'Arousal': 0.0, 'Dominance': 0.0}
    vad_count = 0
    emolex_count = 0
    emoint_count = 0
    total_word_count = len(words)
    
    for word in words:
        emolex = get_emolex_scores(word, is_spanish)
        emoint = get_emoint_scores(word, is_spanish)
        emovad = get_emovad_scores(word, is_spanish)
        
        if emolex:
            emolex_count += 1
            for emotion, score in emolex.items():
                if emotion in all_emotions:
                    emotion_scores[emotion] += score

        if emoint:
            emoint_count += 1
            if emoint['Emotion'] in all_emotions:
                intensity_scores[emoint['Emotion']] += emoint['Emotion-Intensity-Score']

        if emovad:
            vad_scores['Valence'] += emovad['Valence']
            vad_scores['Arousal'] += emovad['Arousal']
            vad_scores['Dominance'] += emovad['Dominance']
            vad_count += 1

    # Ensure no NaN values are present in the calculations
    emotion_scores = {k: 0 if pd.isna(v) else v for k, v in emotion_scores.items()}
    intensity_scores = {k: 0 if pd.isna(v) else v for k, v in intensity_scores.items()}
    vad_scores = {k: 0.0 if pd.isna(v) else v for k, v in vad_scores.items()}

    # Calculate averages for VAD and EmoInt using Fisher's Z transformation
    if vad_count > 0:
        vad_scores_z = {k: fisher_z(vad_scores[k] / vad_count) for k in vad_scores}
        vad_scores_avg = {k: inverse_fisher_z(v) for k, v in vad_scores_z.items()}
    else:
        vad_scores_avg = {k: 0.0 for k in vad_scores}

    if emoint_count > 0:
        intensity_scores_z = {k: fisher_z(intensity_scores[k] / emoint_count) for k in intensity_scores}
        intensity_scores_avg = {k: inverse_fisher_z(v) for k, v in intensity_scores_z.items()}
    else:
        intensity_scores_avg = {k: 0.0 for k in intensity_scores}

    # Calculate averages over total words using Fisher's Z transformation
    vad_scores_total_z = {k: fisher_z(vad_scores[k] / total_word_count) for k in vad_scores}
    vad_scores_total_avg = {f"{k}_total_avg": inverse_fisher_z(v) for k, v in vad_scores_total_z.items()}

    intensity_scores_total_z = {k: fisher_z(intensity_scores[k] / total_word_count) for k in intensity_scores}
    intensity_scores_total_avg = {f"{k}_total_avg": inverse_fisher_z(v) for k, v in intensity_scores_total_z.items()}

    emotion_scores_total_avg = {f"{k}_total_avg": emotion_scores[k] / total_word_count for k in emotion_scores}
    
    return (
        emotion_scores, intensity_scores, vad_scores_avg, 
        emolex_count, total_word_count, 
        vad_scores_total_avg, intensity_scores_total_avg, emotion_scores_total_avg
    )

# Apply aggregation to Processed_Content
(
    emotion_df['Content_Emotion_Scores'],
    emotion_df['Content_Intensity_Scores'],
    emotion_df['Content_VAD_Scores'],
    emotion_df['Content_EmoLex_Words'],
    emotion_df['Content_Total_Words'],
    emotion_df['Content_VAD_Total_Avg'],
    emotion_df['Content_Intensity_Total_Avg'],
    emotion_df['Content_Emotion_Total_Avg']
) = zip(*emotion_df.progress_apply(lambda x: aggregate_content_scores(x['Processed_Content'].split(), x['is_spanish']), axis=1))

# Expand Content_Emotion_Scores into separate columns
emotion_df = pd.concat([emotion_df.drop(['Content_Emotion_Scores'], axis=1), emotion_df['Content_Emotion_Scores'].progress_apply(pd.Series)], axis=1)

# Expand Content_Intensity_Scores into separate columns and add '_intensity' suffix
intensity_scores_df = emotion_df['Content_Intensity_Scores'].progress_apply(pd.Series).add_suffix('_intensity')
emotion_df = pd.concat([emotion_df.drop(['Content_Intensity_Scores'], axis=1), intensity_scores_df], axis=1)

# Expand Content_VAD_Scores into separate columns
vad_scores_df = emotion_df['Content_VAD_Scores'].progress_apply(pd.Series)
emotion_df = pd.concat([emotion_df.drop(['Content_VAD_Scores'], axis=1), vad_scores_df], axis=1)

# Expand Total Averages for VAD, Intensity, and Emotion counts
vad_total_avg_df = emotion_df['Content_VAD_Total_Avg'].progress_apply(pd.Series)
emotion_df = pd.concat([emotion_df.drop(['Content_VAD_Total_Avg'], axis=1), vad_total_avg_df], axis=1)

intensity_total_avg_df = emotion_df['Content_Intensity_Total_Avg'].progress_apply(pd.Series)
emotion_df = pd.concat([emotion_df.drop(['Content_Intensity_Total_Avg'], axis=1), intensity_total_avg_df], axis=1)

emotion_total_avg_df = emotion_df['Content_Emotion_Total_Avg'].progress_apply(pd.Series)
emotion_df = pd.concat([emotion_df.drop(['Content_Emotion_Total_Avg'], axis=1), emotion_total_avg_df], axis=1)

# Save the updated dataframe to a new CSV file
emotion_df.to_csv('data_storage/final_full_sentiment.csv', index=False)

emotion_df

In [ ]:
emotion_df.columns = ["Section", "Title", "Date", "Time", "Authors", "URL", "Content", "is_spanish", "Processed_Title", "Processed_Content", "English_Title", "Translated_Content", "Title_Pos", "Title_Neu", "Title_Neg", "Title_Compound", "Content_Pos", "Content_Neu", "Content_Neg", "Content_Compound", "Title_Sentiment", "Content_Sentiment", "Title_EmoLex_Words", "Title_Total_Words", "Title_anger", "Title_anticipation", "Title_disgust", "Title_fear", "Title_joy", "Title_negative", "Title_positive", "Title_sadness", "Title_surprise", "Title_trust", "Title_anger_intensity", "Title_anticipation_intensity", "Title_disgust_intensity", "Title_fear_intensity", "Title_joy_intensity", "Title_negative_intensity", "Title_positive_intensity", "Title_sadness_intensity", "Title_surprise_intensity", "Title_trust_intensity", "Title_Valence", "Title_Arousal", "Title_Dominance", "Title_Valence_total_avg", "Title_Arousal_total_avg", "Title_Dominance_total_avg", "Title_intensity_anger_total_avg", "Title_intensity_anticipation_total_avg", "Title_intensity_disgust_total_avg", "Title_intensity_fear_total_avg", "Title_intensity_joy_total_avg", "Title_intensity_negative_total_avg", "Title_intensity_positive_total_avg", "Title_intensity_sadness_total_avg", "Title_intensity_surprise_total_avg", "Title_intensity_trust_total_avg", "Title_anger_total_avg", "Title_anticipation_total_avg", "Title_disgust_total_avg", "Title_fear_total_avg", "Title_joy_total_avg", "Title_negative_total_avg", "Title_positive_total_avg", "Title_sadness_total_avg", "Title_surprise_total_avg", "Title_trust_total_avg", "Content_EmoLex_Words", "Content_Total_Words", "Content_anger", "Content_anticipation", "Content_disgust", "Content_fear", "Content_joy", "Content_negative", "Content_positive", "Content_sadness", "Content_surprise", "Content_trust", "Content_anger_intensity", "Content_anticipation_intensity", "Content_disgust_intensity", "Content_fear_intensity", "Content_joy_intensity", "Content_negative_intensity", "Content_positive_intensity", "Content_sadness_intensity", "Content_surprise_intensity", "Content_trust_intensity", "Content_Valence", "Content_Arousal", "Content_Dominance", "Content_Valence_total_avg", "Content_Arousal_total_avg", "Content_Dominance_total_avg", "Content_intensity_anger_total_avg", "Content_intensity_anticipation_total_avg", "Content_intensity_disgust_total_avg", "Content_intensity_fear_total_avg", "Content_intensity_joy_total_avg", "Content_intensity_negative_total_avg", "Content_intensity_positive_total_avg", "Content_intensity_sadness_total_avg", "Content_intensity_surprise_total_avg", "Content_intensity_trust_total_avg", "Content_anger_total_avg", "Content_anticipation_total_avg", "Content_disgust_total_avg", "Content_fear_total_avg", "Content_joy_total_avg", "Content_negative_total_avg", "Content_positive_total_avg", "Content_sadness_total_avg", "Content_surprise_total_avg", "Content_trust_total_avg"]
emotion_df.to_csv('data_storage/final_full_sentiment.csv', index=False)
emotion_df

## Keyword Analysis

In [ ]:
import pandas as pd
from tqdm import tqdm
import re

# Load the data
keyword_df = pd.read_csv('data_storage/final_full_sentiment.csv')
keyword_df['Authors'] = keyword_df['Authors'].apply(clean_authors)
keyword_df['Section'] = keyword_df['Section'].apply(clean_authors)

# Define the keywords
keywords = [
    "asian", "hispanic", "latinx", "diversity", "equity", "inclusion", "equality",
    "lgbtq", "lgbtqia", "queer", "middle eastern", "black", "gay", "transgender",
    "bisexual", "lesbian", "pansexual", "asexual", "homosexual", "non-binary",
    "intersex", "aromantic", "cisgender", "coming out", "genderfluid", "privilege",
    "pronouns", "undocumented", "neurodiversity", "neurodivergent", "disability",
    "microaggression", "identity", "discrimination", "culture", "misgendering", "masculinity", "sexuality"
]

# Define the words to exclude if they follow "black" or precede "culture"
exclude_black = [
    "pepper", "hole", "coffee", "sesame", "box", "beans", "market", "bean", "holes", 
    "salt", "square", "mold", "nissan", "tea", "screens", "screen", "garlic", "shoes", 
    "bear", "ski", "puffer", "milk", "smoke", "pants", "clothing", "bicycle", "gloves", 
    "ops", "suit", "liquid", "sweatshirt", "belt", "fur", "sugar", "swan", "seeds", 
    "lipstick", "levi", "stretching", "olives", "spoiler", "guitar", "rice", "tahini", 
    "bike", "silhouette", "swimsuit", "backpacks", "page", "sedan", "corset", "skinny", 
    "dress", "mask", "mesh", "drape", "binder", "motorcycle", "rubber", "clothes", "dots", 
    "cats", "toyota", "mascara", "designer", "colors", "honda", "chevrolet", "4runner", 
    "long", "shirt", "sauce", "birds", "light", "boots", "pavement", "phone", "sand", 
    "eye", "ground", "landscape", "lace", "backpack", "charcoal", "jeans", "hoodie", 
    "bicycles", "truffle", "leather", "bmw", "denim", "attire", "jumpsuit", "raspberry", 
    "tunnel", "car", "compact", "dildo", "sweater", "sunglasses"
]

exclude_culture = ["lab", "laboratory", "cell"]

# Function to count keyword occurrences, considering exclusions
def count_keywords(row):
    if row['is_spanish']:
        title = row['English_Title'].lower() if row['English_Title'] else ""
        content = row['Translated_Content'].lower() if row['Translated_Content'] else ""
    else:
        title = row['Processed_Title'].lower() if row['Processed_Title'] else ""
        content = row['Processed_Content'].lower() if row['Processed_Content'] else ""
    
    title_count = 0
    content_count = 0
    
    # Count keywords in title
    for keyword in keywords:
        if keyword in title:
            if keyword == "black":
                # Check if excluded words follow "black"
                if any(re.search(rf"\bblack\s+{excl}\b", title) for excl in exclude_black):
                    continue
                title_count += 1
            elif keyword == "culture":
                # Check if excluded words precede "culture"
                if any(re.search(rf"\b{excl}\s+culture\b", title) for excl in exclude_culture):
                    continue
                title_count += 1
            else:
                title_count += 1

    # Count keywords in content
    for keyword in keywords:
        if keyword in content:
            if keyword == "black":
                # Check if excluded words follow "black"
                if any(re.search(rf"\bblack\s+{excl}\b", content) for excl in exclude_black):
                    continue
                content_count += 1
            elif keyword == "culture":
                # Check if excluded words precede "culture"
                if any(re.search(rf"\b{excl}\s+culture\b", content) for excl in exclude_culture):
                    continue
                content_count += 1
            else:
                content_count += 1

    return pd.Series([title_count, content_count], index=['Title_Keyword_Count', 'Content_Keyword_Count'])

# Add a progress bar to the apply function
tqdm.pandas()

# Apply the function to each row with a progress bar
keyword_df[['Title_Keyword_Count', 'Content_Keyword_Count']] = keyword_df.progress_apply(count_keywords, axis=1)

# Create Yes/No columns based on the counts
keyword_df['Title_Keyword_Flag'] = keyword_df['Title_Keyword_Count'].apply(lambda x: 'Yes' if x > 0 else 'No')
keyword_df['Content_Keyword_Flag'] = keyword_df['Content_Keyword_Count'].apply(lambda x: 'Yes' if x > 0 else 'No')

# Save the updated dataframe to a new CSV file
keyword_df.to_csv('data_storage/full_sentiment_keywords.csv', index=False)

keyword_df

## Chart Preparation

#### Keywords

In [ ]:
final_df = pd.read_csv('data_storage/full_sentiment_keywords.csv')
final_df['Authors'] = final_df['Authors'].apply(clean_authors)
final_df['Section'] = final_df['Section'].apply(clean_authors)

final_df

In [ ]:
# Ensure the 'Date' column is in datetime format, then extract the year
final_df['Date'] = pd.to_datetime(final_df['Date'])
final_df['Year'] = final_df['Date'].dt.year

# Create a combined keyword flag
final_df['Keyword_Flag'] = final_df.apply(
    lambda row: 'Yes' if row['Title_Keyword_Flag'] == 'Yes' or row['Content_Keyword_Flag'] == 'Yes' else 'No',
    axis=1
)

# Group by year and aggregate the data
grouped_keywords_df = final_df.groupby('Year').agg(
    Article_Count=('URL', 'count'),
    Title_Keyword_Count=('Title_Keyword_Count', 'sum'),
    Content_Keyword_Count=('Content_Keyword_Count', 'sum'),
    Title_Flag_Yes_Count=('Title_Keyword_Flag', lambda x: (x == 'Yes').sum()),
    Title_Flag_No_Count=('Title_Keyword_Flag', lambda x: (x == 'No').sum()),
    Content_Flag_Yes_Count=('Content_Keyword_Flag', lambda x: (x == 'Yes').sum()),
    Content_Flag_No_Count=('Content_Keyword_Flag', lambda x: (x == 'No').sum()),
    Keyword_Flag_Yes_Count=('Keyword_Flag', lambda x: (x == 'Yes').sum()),
    Keyword_Flag_No_Count=('Keyword_Flag', lambda x: (x == 'No').sum())
).reset_index()

# Calculate total keyword count and average keywords per article
grouped_keywords_df['Total_Keyword_Count'] = grouped_keywords_df['Title_Keyword_Count'] + grouped_keywords_df['Content_Keyword_Count']
grouped_keywords_df['Average_Keyword_Per_Article'] = grouped_keywords_df['Total_Keyword_Count'] / grouped_keywords_df['Article_Count']

# Proportion of articles with a keyword overall
grouped_keywords_df['Keyword_Flag_Proportion'] = grouped_keywords_df['Keyword_Flag_Yes_Count'] / grouped_keywords_df['Article_Count']

# Do title proportion
grouped_keywords_df['Title_Flag_Proportion'] = grouped_keywords_df['Title_Flag_Yes_Count'] / grouped_keywords_df['Article_Count']

# Save keyword proportion to csv file
grouped_keywords_df[['Year', 'Keyword_Flag_Proportion']].to_csv('chart_storage/proportion_of_articles_with_keywords_by_year.csv', index=False)

# Display the resulting dataframe
grouped_keywords_df


In [ ]:
final_df = pd.read_csv('data_storage/full_sentiment_keywords.csv')

# Ensure the 'Date' column is in datetime format, then extract the year and month
final_df['Date'] = pd.to_datetime(final_df['Date'])
final_df['Year'] = final_df['Date'].dt.year
final_df['Month'] = final_df['Date'].dt.month

# Create a combined keyword flag
final_df['Keyword_Flag'] = final_df.apply(
    lambda row: 'Yes' if row['Title_Keyword_Flag'] == 'Yes' or row['Content_Keyword_Flag'] == 'Yes' else 'No',
    axis=1
)

# Group by year and month and aggregate the data
grouped_keywords_df = final_df.groupby(['Year', 'Month']).agg(
    Article_Count=('URL', 'count'),
    Title_Keyword_Count=('Title_Keyword_Count', 'sum'),
    Content_Keyword_Count=('Content_Keyword_Count', 'sum'),
    Title_Flag_Yes_Count=('Title_Keyword_Flag', lambda x: (x == 'Yes').sum()),
    Title_Flag_No_Count=('Title_Keyword_Flag', lambda x: (x == 'No').sum()),
    Content_Flag_Yes_Count=('Content_Keyword_Flag', lambda x: (x == 'Yes').sum()),
    Content_Flag_No_Count=('Content_Keyword_Flag', lambda x: (x == 'No').sum()),
    Keyword_Flag_Yes_Count=('Keyword_Flag', lambda x: (x == 'Yes').sum()),
    Keyword_Flag_No_Count=('Keyword_Flag', lambda x: (x == 'No').sum())
).reset_index()

# Calculate total keyword count and average keywords per article
grouped_keywords_df['Total_Keyword_Count'] = grouped_keywords_df['Title_Keyword_Count'] + grouped_keywords_df['Content_Keyword_Count']
grouped_keywords_df['Average_Keyword_Per_Article'] = grouped_keywords_df['Total_Keyword_Count'] / grouped_keywords_df['Article_Count']

# Proportion of articles with a keyword overall
grouped_keywords_df['Keyword_Flag_Proportion'] = grouped_keywords_df['Keyword_Flag_Yes_Count'] / grouped_keywords_df['Article_Count']

# Proportion of articles with keywords in title
grouped_keywords_df['Title_Flag_Proportion'] = grouped_keywords_df['Title_Flag_Yes_Count'] / grouped_keywords_df['Article_Count']

# Save keyword proportion to CSV file by month
grouped_keywords_df[['Year', 'Month', 'Keyword_Flag_Proportion']].to_csv('chart_storage/proportion_of_articles_with_keywords_by_month.csv', index=False)

# Filter
grouped_keywords_df = grouped_keywords_df[['Year', 'Month', 'Keyword_Flag_Proportion']][(grouped_keywords_df['Year'] >= 2022)]

# Drop last row
grouped_keywords_df = grouped_keywords_df[:-1]

# Save keyword proportion to CSV file by month
grouped_keywords_df.to_csv('chart_storage/modern_proportion_of_articles_with_keywords_by_month.csv', index=False)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 120)

# Display the resulting dataframe
grouped_keywords_df

In [ ]:
keywords_month = pd.read_csv('chart_storage/proportion_of_articles_with_keywords_by_month.csv')
keywords_month = keywords_month.groupby('Month').agg({'Keyword_Flag_Proportion': 'mean'}).reset_index()

# Save to a new CSV file
keywords_month.to_csv('chart_storage/average_keyword_proportion_by_month.csv', index=False)

In [ ]:
inal_df = pd.read_csv('data_storage/full_sentiment_keywords.csv')

# Ensure the 'Date' column is in datetime format, then extract the year and month
final_df['Date'] = pd.to_datetime(final_df['Date'])
final_df['Year'] = final_df['Date'].dt.year
final_df['Month'] = final_df['Date'].dt.month

# Define the keywords
keywords = [
    "asian", "hispanic", "latinx", "diversity", "equity", "inclusion", "equality",
    "lgbtq", "lgbtqia", "queer", "middle eastern", "black", "gay", "transgender",
    "bisexual", "lesbian", "pansexual", "asexual", "homosexual", "non-binary",
    "intersex", "aromantic", "cisgender", "coming out", "genderfluid", "privilege",
    "pronouns", "undocumented", "neurodiversity", "neurodivergent", "disability",
    "microaggression", "identity", "discrimination", "culture", "misgendering", "masculinity", "sexuality"
]

# Function to find keywords in the content
def find_keywords(content, keywords):
    found_keywords = [kw for kw in keywords if kw in content]
    return found_keywords

# Apply the function to the content column
final_df['Content_Keywords_List'] = final_df['Content'].apply(lambda x: find_keywords(x, keywords) if pd.notnull(x) else [])

# Display rows where the Content_Keyword_Flag is 'Yes'
final_df_with_keywords = final_df[final_df['Content_Keyword_Flag'] == 'Yes'][['Content', 'Content_Keywords_List', 'Year', 'Month']]

# Use pd series to get content keywords list stack
final_df_with_keywords = final_df_with_keywords.set_index('Content').apply(lambda x: x.apply(pd.Series).stack()).reset_index()

# filter month as 6 and year as 2024
final_df_with_keywords = final_df_with_keywords[(final_df_with_keywords['Year'] == 2024) & (final_df_with_keywords['Month'] == 6)]

final_df_with_keywords['Content_Keywords_List'].value_counts()

In [ ]:
final_df[final_df['Content_Keyword_Flag'] == 'Yes'][['Content', 'Content_Keywords_List', 'Year', 'Month']][final_df['Year'] == 2024][final_df['Month'] == 6]

In [ ]:
final_df

In [ ]:
# Define the keywords
keywords = [
    "asian", "hispanic", "latinx", "diversity", "equity", "inclusion", "equality",
    "lgbtq", "lgbtqia", "queer", "middle eastern", "black", "gay", "transgender",
    "bisexual", "lesbian", "pansexual", "asexual", "homosexual", "non-binary",
    "intersex", "aromantic", "cisgender", "coming out", "genderfluid", "privilege",
    "pronouns", "undocumented", "neurodiversity", "neurodivergent", "disability",
    "microaggression", "identity", "discrimination", "culture", "misgendering", "masculinity", "sexuality"
]

# Function to find all words preceding "diversity" in the content
def find_words_before_diversity(content):
    matches = re.findall(r'\b(\w+)\b(?=\s+identity\b)', content.lower())
    return matches

# Apply the function to the content column
final_df['Words_Before_Diversity'] = final_df['Content'].apply(lambda x: find_words_before_diversity(x) if pd.notnull(x) else [])

# Filter out rows where there are words before "diversity"
filtered_df = final_df[final_df['Words_Before_Diversity'].apply(len) > 0]

# Explode the 'Words_Before_Diversity' column to have one entry per row
exploded_df = filtered_df.explode('Words_Before_Diversity')

# Count occurrences of each word preceding "diversity"
words_before_diversity_counts = exploded_df['Words_Before_Diversity'].value_counts().reset_index()
words_before_diversity_counts.columns = ['Word', 'Count']

words_before_diversity_counts['']

#### Title Sentiment

In [ ]:
# Load the data
final_df = pd.read_csv('data_storage/full_sentiment_keywords.csv')
final_df['Authors'] = final_df['Authors'].apply(clean_authors)
final_df['Section'] = final_df['Section'].apply(clean_authors)
final_df['Year'] = final_df['Date'].str[:4].astype(int)

# Group title sentiment by year
grouped_title_sentiment_df = final_df.groupby('Year').agg(
    Article_Count=('URL', 'count'),
    Title_Pos_Count=('Title_Sentiment', lambda x: (x == 'Positive').sum()),
    Title_Neu_Count=('Title_Sentiment', lambda x: (x == 'Neutral').sum()),
    Title_Neg_Count=('Title_Sentiment', lambda x: (x == 'Negative').sum())
).reset_index() 

# Calculate proportions
grouped_title_sentiment_df['Title_Pos_Proportion'] = grouped_title_sentiment_df['Title_Pos_Count'] / grouped_title_sentiment_df['Article_Count']
grouped_title_sentiment_df['Title_Neu_Proportion'] = grouped_title_sentiment_df['Title_Neu_Count'] / grouped_title_sentiment_df['Article_Count']
grouped_title_sentiment_df['Title_Neg_Proportion'] = grouped_title_sentiment_df['Title_Neg_Count'] / grouped_title_sentiment_df['Article_Count']

# Drop all keywords except Year and Proportion
grouped_title_sentiment_df = grouped_title_sentiment_df[['Year', 'Title_Pos_Proportion', 'Title_Neu_Proportion', 'Title_Neg_Proportion']]

# Save to a new CSV file
grouped_title_sentiment_df.to_csv('chart_storage/all_title_overall_sentiment_by_year.csv', index=False)

grouped_title_sentiment_df

In [ ]:
# Load the data
final_df = pd.read_csv('data_storage/full_sentiment_keywords.csv')
final_df['Authors'] = final_df['Authors'].apply(clean_authors)
final_df['Section'] = final_df['Section'].apply(clean_authors)
final_df['Year'] = final_df['Date'].str[:4].astype(int)

final_title_yes_df = final_df[final_df['Title_Keyword_Flag'] == 'Yes']

# Group title sentiment by year
grouped_title_sentiment_df = final_title_yes_df.groupby('Year').agg(
    Article_Count=('URL', 'count'),
    Title_Pos_Count=('Title_Sentiment', lambda x: (x == 'Positive').sum()),
    Title_Neu_Count=('Title_Sentiment', lambda x: (x == 'Neutral').sum()),
    Title_Neg_Count=('Title_Sentiment', lambda x: (x == 'Negative').sum())
).reset_index() 

# Calculate proportions
grouped_title_sentiment_df['Title_Pos_Proportion'] = grouped_title_sentiment_df['Title_Pos_Count'] / grouped_title_sentiment_df['Article_Count']
grouped_title_sentiment_df['Title_Neu_Proportion'] = grouped_title_sentiment_df['Title_Neu_Count'] / grouped_title_sentiment_df['Article_Count']
grouped_title_sentiment_df['Title_Neg_Proportion'] = grouped_title_sentiment_df['Title_Neg_Count'] / grouped_title_sentiment_df['Article_Count']

# Drop all keywords except Year and Proportion
grouped_title_sentiment_df = grouped_title_sentiment_df[['Year', 'Title_Pos_Proportion', 'Title_Neu_Proportion', 'Title_Neg_Proportion']]

# Save to a new CSV file
grouped_title_sentiment_df.to_csv('chart_storage/dei_title_overall_sentiment_by_year.csv', index=False)

grouped_title_sentiment_df

#### Content Sentiment


In [ ]:
# Load the data
final_df = pd.read_csv('data_storage/full_sentiment_keywords.csv')
final_df['Authors'] = final_df['Authors'].apply(clean_authors)
final_df['Section'] = final_df['Section'].apply(clean_authors)
final_df['Year'] = final_df['Date'].str[:4].astype(int)
final_df['Month'] = final_df['Date'].str[5:7].astype(int)

# Number of total emotion words as a dataframe for each emotion by year
emotion_words_df = final_df.groupby(['Year', 'Month']).agg(
    title_anger=('Title_anger', 'sum'),
    title_anticipation=('Title_anticipation', 'sum'),
    title_disgust=('Title_disgust', 'sum'),
    title_fear=('Title_fear', 'sum'),
    title_joy=('Title_joy', 'sum'),
    title_sadness=('Title_sadness', 'sum'),
    title_surprise=('Title_surprise', 'sum'),
    title_trust=('Title_trust', 'sum'),
    title_words=('Title_Total_Words', 'sum'),
    content_anger=('Content_anger', 'sum'),
    content_anticipation=('Content_anticipation', 'sum'),
    content_disgust=('Content_disgust', 'sum'),
    content_fear=('Content_fear', 'sum'),
    content_joy=('Content_joy', 'sum'),
    content_sadness=('Content_sadness', 'sum'),
    content_surprise=('Content_surprise', 'sum'),
    content_trust=('Content_trust', 'sum'),
    content_words=('Content_Total_Words', 'sum')
).reset_index()

# Group emotion words by year
emotion_words_df_year = emotion_words_df.groupby('Year').sum().drop('Month', axis=1).reset_index()
emotion_words_df_year

In [ ]:
# Load data
final_df = pd.read_csv('data_storage/full_sentiment_keywords.csv')
final_df['Authors'] = final_df['Authors'].apply(clean_authors)
final_df['Section'] = final_df['Section'].apply(clean_authors)
final_df['Year'] = final_df['Date'].str[:4].astype(int)
final_df['Month'] = final_df['Date'].str[5:7].astype(int)

# Define the relevant emotion columns for title and content
title_columns = [
    'Title_anger', 'Title_anticipation', 'Title_disgust', 'Title_fear', 
    'Title_joy', 'Title_sadness', 
    'Title_surprise', 'Title_trust'
]

content_columns = [
    'Content_anger', 'Content_anticipation', 'Content_disgust', 'Content_fear', 
    'Content_joy', 'Content_sadness', 
    'Content_surprise', 'Content_trust'
]

# Calculate proportions and create new columns in a new DataFrame
proportions_df = final_df.copy()
for col in title_columns:
    proportions_df[col + '_Proportion'] = proportions_df[col] / proportions_df['Title_Total_Words']

for col in content_columns:
    proportions_df[col + '_Proportion'] = proportions_df[col] / proportions_df['Content_Total_Words']

# Content Emotion Proportions
prop_content = proportions_df[['Year'] + content_columns + ['Content_Total_Words'] + [col + '_Proportion' for col in content_columns]].groupby('Year').mean()
prop_content = prop_content.drop(columns=['Content_anger', 'Content_anticipation', 'Content_disgust', 'Content_fear', 'Content_joy', 'Content_sadness', 'Content_surprise', 'Content_trust', 'Content_Total_Words'])
prop_content.reset_index(inplace=True)

# Title Emotion Proportions
prop_title = proportions_df[['Year'] + title_columns + ['Title_Total_Words'] + [col + '_Proportion' for col in title_columns]].groupby('Year').mean()
prop_title = prop_title.drop(columns=['Title_anger', 'Title_anticipation', 'Title_disgust', 'Title_fear', 'Title_joy', 'Title_sadness', 'Title_surprise', 'Title_trust', 'Title_Total_Words'])
prop_title.reset_index(inplace=True)

# Concatenate the two dataframes and drop the year duplicate
prop_emotion = pd.concat([prop_title, prop_content], axis=1)
prop_emotion = prop_emotion.loc[:, ~prop_emotion.columns.duplicated()]
prop_emotion

# Save the updated dataframe to a new CSV file
prop_emotion.to_csv('chart_storage/avg_emotion_proportion_by_year.csv', index=False)

# Display the resulting dataframe
prop_emotion

In [ ]:
# Load data
final_df = pd.read_csv('data_storage/full_sentiment_keywords.csv')
final_df['Authors'] = final_df['Authors'].apply(clean_authors)
final_df['Section'] = final_df['Section'].apply(clean_authors)
final_df['Year'] = final_df['Date'].str[:4].astype(int)
final_df['Month'] = final_df['Date'].str[5:7].astype(int)

# Define the relevant emotion columns for title and content
title_columns = [
    'Title_anger', 'Title_anticipation', 'Title_disgust', 'Title_fear', 
    'Title_joy', 'Title_sadness', 
    'Title_surprise', 'Title_trust'
]

content_columns = [
    'Content_anger', 'Content_anticipation', 'Content_disgust', 'Content_fear', 
    'Content_joy', 'Content_sadness', 
    'Content_surprise', 'Content_trust'
]

final_df['Keyword_Flag'] = final_df.apply(
    lambda row: 'Yes' if row['Title_Keyword_Flag'] == 'Yes' or row['Content_Keyword_Flag'] == 'Yes' else 'No',
    axis=1
)

# Calculate proportions and create new columns in a new DataFrame
proportions_df = final_df[final_df['Keyword_Flag'] == 'Yes'].copy()
for col in title_columns:
    proportions_df[col + '_Proportion'] = proportions_df[col] / proportions_df['Title_Total_Words']

for col in content_columns:
    proportions_df[col + '_Proportion'] = proportions_df[col] / proportions_df['Content_Total_Words']

# Content Emotion Proportions
prop_content = proportions_df[['Year'] + content_columns + ['Content_Total_Words'] + [col + '_Proportion' for col in content_columns]].groupby('Year').mean()
prop_content = prop_content.drop(columns=['Content_anger', 'Content_anticipation', 'Content_disgust', 'Content_fear', 'Content_joy', 'Content_sadness', 'Content_surprise', 'Content_trust', 'Content_Total_Words'])
prop_content.reset_index(inplace=True)

# Title Emotion Proportions
prop_title = proportions_df[['Year'] + title_columns + ['Title_Total_Words'] + [col + '_Proportion' for col in title_columns]].groupby('Year').mean()
prop_title = prop_title.drop(columns=['Title_anger', 'Title_anticipation', 'Title_disgust', 'Title_fear', 'Title_joy', 'Title_sadness', 'Title_surprise', 'Title_trust', 'Title_Total_Words'])
prop_title.reset_index(inplace=True)

# Concatenate the two dataframes and drop the year duplicate
prop_emotion = pd.concat([prop_title, prop_content], axis=1)
prop_emotion = prop_emotion.loc[:, ~prop_emotion.columns.duplicated()]
prop_emotion

# Save the updated dataframe to a new CSV file
prop_emotion.to_csv('chart_storage/dei_avg_emotion_proportion_by_year.csv', index=False)

# Display the resulting dataframe
prop_emotion

#### Extra Analysis

In [ ]:
# Load data
final_df = pd.read_csv('data_storage/full_sentiment_keywords.csv')
final_df['Authors'] = final_df['Authors'].apply(clean_authors)
final_df['Section'] = final_df['Section'].apply(clean_authors)
final_df['Year'] = final_df['Date'].str[:4].astype(int)


# Pull title, title pos, title neu, title neg, title compound, title sentiment
title_sentiment_df = final_df[['Year', 'Title', 'Title_Pos', 'Title_Neu', 'Title_Neg', 'Title_Compound', 'Title_Sentiment']]


for title in title_sentiment_df[title_sentiment_df['Title_Sentiment'] == "Negative"].head(15)['Title']:
    print(title)

title_sentiment_df[title_sentiment_df['Title_Sentiment'] == "Negative"].head(15)

In [ ]:
final_df

## Exploratory Data Analysis

In [ ]:
# Explode the 'Section' column to have one row per section per article
final_df_copy = final_df.explode('Section')

# Now, group by 'Year' and 'Section' and count the number of articles
section_df = final_df_copy.groupby(['Year', 'Section'])['URL'].count().reset_index()
section_df.columns = ['Year', 'Section', 'Article_Count']

# Display the resulting dataframe
section_df[section_df['Year'] == 2024]

In [ ]:
combined_df['Authors'].value_counts()

In [ ]:
combined_df['Authors'].apply(pd.Series).stack().value_counts()

In [ ]:
combined_df['Section'].value_counts()

In [ ]:
sections = combined_df['Section'].apply(pd.Series).stack().value_counts()
sections

In [ ]:
import pandas as pd
final_df = pd.read_csv('data_storage/full_sentiment_keywords.csv')
final_df['Year'] = final_df['Date'].str[:4].astype(int)

In [ ]:
send_1 = pd.read_csv('data_storage/israel_palestine.csv')
# convert date to datetime
send_1['Date'] = pd.to_datetime(send_1['Date'])

# give me df where IP_Title_Keyword_Flaf is Yes and date >= 10/7/2023

send_1 = send_1[(send_1['IP_Title_Keyword_Flag'] == 'Yes') & (send_1['Date'] >= '2023-10-07')]
send_1[['Section', 'Title', 'Authors', 'Date', 'URL', 'Title_Sentiment']].to_csv('chart_storage/Bye Bye/ip.csv', index=False)

In [ ]:
send_2 = pd.read_csv('data_storage/full_sentiment_keywords.csv')
send_2['Date'] = pd.to_datetime(send_2['Date']) 
send_2 = send_2[(send_2['Title_Keyword_Flag'] == 'Yes') & (send_2['Date'] >= '2023-06-10')]
# clean section
send_2['Section'] = send_2['Section'].apply(clean_authors)
# Section is a list - see if news is in the list
array = []
for element in send_2['Section']:
    if 'News' in element:
        array.append(True)
    else:
        array.append(False)
send_2['array'] = array
send_2 = send_2[send_2['array'] == True]
send_2[['Section', 'Title', 'Authors', 'Date', 'URL', 'Title_Sentiment']].to_csv('chart_storage/Bye Bye/news.csv', index=False)
send_2[['Section', 'Title', 'Authors', 'Date', 'URL', 'Title_Sentiment']]